# Magnetic Avalanche Simulation

Simulates the **magnetic reconnection cascade** (the 'magnetic avalanche'
captured by Solar Orbiter on 30 Sept 2024) as a UTAC phase transition.

**Reference:** Solar Orbiter / A&A 2026 — close-pass flare ribbon capture.

## Simulation
- Active region energy buildup over days/weeks
- Reconnection threshold crossing → eruptive phase
- CREP Γ gating: only erupts when Γ > θ_PT
- Geomagnetic storm impact cascade

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from solar_flare_utac.active_region import MagneticActiveRegion
from solar_flare_utac.reconnection import ReconnectionThreshold
from solar_flare_utac.crep_solar import SolarCREP
from solar_flare_utac.geomagnetic import GeomagneticStorm
from solar_flare_utac.constants import H_THRESHOLD, GAMMA_SOLAR, H_STAR_QUIET

# Setup
ar = MagneticActiveRegion(r_buildup=0.06, h_init=0.10, seed=42)
rc = ReconnectionThreshold()
crep = SolarCREP(seed=42)
geo = GeomagneticStorm(seed=42)

# Simulate 7 days with manually elevated Γ after day 5 (pre-flare CREP buildup)
dt = 0.1
duration = 168.0  # 7 days
n = int(duration / dt)

H_history = [ar.H]
Gamma_history = []
lam_history = []
flare_times = []

for i in range(n):
    t = i * dt
    H = ar.H
    # Gradually increase Gamma after t=120h (simulating pre-flare CREP buildup)
    Gamma = GAMMA_SOLAR * (1 + 2.5 * max(0, (t - 120) / 48))
    lam = rc.reconnection_rate(H, Gamma)
    result = ar.step(dt_hours=dt, lambda_reconnect=lam, noise_sigma=0.001)
    H_history.append(ar.H)
    Gamma_history.append(Gamma)
    lam_history.append(lam)
    if result['flare_triggered']:
        flare_times.append(t)
        print(f't={t:.1f}h: FLARE! E={result["energy_released_J"]:.2e} J, '
              f'Gamma={Gamma:.4f}')

times = np.arange(len(H_history)) * dt
print(f'\nTotal flares: {len(flare_times)}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(times, H_history, lw=1.0, color='orangered', label='H(t)')
axes[0].axhline(H_THRESHOLD, ls='--', color='red', lw=0.8, label='H_threshold')
axes[0].axhline(H_STAR_QUIET, ls='--', color='blue', lw=0.8, label='H* quiet')
for ft in flare_times:
    axes[0].axvline(ft, color='gold', alpha=0.9, lw=1.5)
axes[0].set_ylabel('H (free energy)')
axes[0].set_title('Magnetic Free Energy Evolution — UTAC Flare Cascade')
axes[0].legend(fontsize=8)

axes[1].plot(np.arange(len(Gamma_history)) * dt, Gamma_history,
             lw=0.8, color='purple', label='Γ(t)')
axes[1].axhline(GAMMA_SOLAR, ls='--', color='black', lw=0.8,
                label=f'θ_PT = Γ_solar = {GAMMA_SOLAR:.4f}')
axes[1].set_ylabel('Γ (CREP)')
axes[1].legend(fontsize=8)

axes[2].semilogy(np.arange(len(lam_history)) * dt, lam_history,
                 lw=0.8, color='teal', label='λ_reconnect')
axes[2].set_ylabel('λ [hr⁻¹]')
axes[2].set_xlabel('Time [hours]')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Geomagnetic storm profile for the largest simulated flare
# Estimate flare energy from a typical X-class event
from solar_flare_utac.constants import E_MAX_J, H_STAR_QUIET

flare_energy_J = (H_THRESHOLD - H_STAR_QUIET) * E_MAX_J
storm = geo.predict_dst(flare_energy_J=flare_energy_J, cme_speed_km_s=1500.0)
print('Storm prediction:')
for k, v in storm.items():
    print(f'  {k}: {v}')

profile = geo.simulate_storm_profile(storm['dst_peak_nT'], duration_hours=96.0)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(profile['times_h'], profile['dst_nT'], lw=1.5, color='navy')
ax.axhline(-50, ls='--', color='orange', label='Moderate storm')
ax.axhline(-100, ls='--', color='red', label='Intense storm')
ax.fill_between(profile['times_h'], profile['dst_nT'], 0,
                where=profile['dst_nT'] < 0, alpha=0.2, color='navy')
ax.set_xlabel('Time [hours after CME arrival]')
ax.set_ylabel('Dst [nT]')
ax.set_title(f'Geomagnetic Storm Profile — {storm["storm_class"]} storm'
             f' (Dst_peak = {storm["dst_peak_nT"]:.0f} nT)')
ax.legend()
plt.tight_layout()
plt.show()